<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_DeepLearning/student/Tutorial4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Tutorial 4: Fully Connected Networks vs CNNs (MNIST)

**Session 1: Deep Learning Foundations**

**Objective:** Motivate convolutional neural networks via inductive bias.


## Tutorial Objectives

Fully connected networks treat an image as a vector. CNNs preserve spatial structure by using local filters shared across the image.

In this tutorial, we will compare:

- An MLP trained on flattened MNIST images.
- A CNN trained on MNIST images.
- Parameter count, training loss, generalization, and learned filters.


In [ ]:
# Imports and shared settings
import random
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
def get_mnist_loaders(batch_size=128, train_subset=12000, val_size=2000):
    """Return small MNIST train/validation/test loaders for quick tutorials."""
    transform = transforms.ToTensor()

    full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    if train_subset is not None:
        indices = torch.randperm(len(full_train))[:train_subset + val_size]
        full_train = Subset(full_train, indices)

    train_size = len(full_train) - val_size
    train_data, val_data = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_mnist_loaders()
images, labels = next(iter(train_loader))
print('Image batch:', images.shape)
print('Label batch:', labels.shape)


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def fit(model, train_loader, val_loader, n_epochs=5, lr=1e-2, momentum=0.0):
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'Epoch {epoch + 1:02d} | '
            f'train loss {train_loss:.3f}, acc {train_acc:.3f} | '
            f'val loss {val_loss:.3f}, acc {val_acc:.3f}'
        )

    return history


def plot_history(history, title='Training curves'):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))

    axs[0].plot(epochs, history['train_loss'], marker='o', label='train')
    axs[0].plot(epochs, history['val_loss'], marker='o', label='validation')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Cross-entropy loss')
    axs[0].set_title('Loss')
    axs[0].legend()

    axs[1].plot(epochs, history['train_acc'], marker='o', label='train')
    axs[1].plot(epochs, history['val_acc'], marker='o', label='validation')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].set_ylim(0, 1)
    axs[1].set_title('Accuracy')
    axs[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Define an MLP and a CNN

The MLP has no built-in assumption about neighboring pixels. The CNN assumes local features can appear in many spatial locations.


In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self, hidden_units=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, x):
        return self.net(x)


class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

mlp = MNISTMLP()
cnn = MNISTCNN()
print(f'MLP parameters: {count_parameters(mlp):,}')
print(f'CNN parameters: {count_parameters(cnn):,}')


## Train Both Models

Both models see the same data. The difference is architectural bias: flattened pixels versus convolution and pooling.


In [ ]:
torch.manual_seed(SEED)
mlp = MNISTMLP()
mlp_history = fit(mlp, train_loader, val_loader, n_epochs=5, lr=0.1)

torch.manual_seed(SEED)
cnn = MNISTCNN()
cnn_history = fit(cnn, train_loader, val_loader, n_epochs=5, lr=0.1, momentum=0.9)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
epochs = np.arange(1, len(mlp_history['train_loss']) + 1)

axs[0].plot(epochs, mlp_history['train_loss'], marker='o', label='MLP train')
axs[0].plot(epochs, cnn_history['train_loss'], marker='o', label='CNN train')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Cross-entropy loss')
axs[0].set_title('Training loss')
axs[0].legend()

axs[1].plot(epochs, mlp_history['val_acc'], marker='o', label='MLP val')
axs[1].plot(epochs, cnn_history['val_acc'], marker='o', label='CNN val')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Validation accuracy')
axs[1].set_ylim(0, 1)
axs[1].set_title('Validation accuracy')
axs[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
loss_fn = nn.CrossEntropyLoss()
mlp_test_loss, mlp_test_acc = evaluate(mlp, test_loader, loss_fn)
cnn_test_loss, cnn_test_acc = evaluate(cnn, test_loader, loss_fn)
print(f'MLP test accuracy: {mlp_test_acc:.3f}')
print(f'CNN test accuracy: {cnn_test_acc:.3f}')


## Visualize Learned CNN Filters

The first convolutional layer learns small image filters. On MNIST, these often look like stroke or edge detectors.


In [ ]:
first_conv = cnn.features[0]
weights = first_conv.weight.detach().cpu()  # shape: out_channels x in_channels x K x K

fig, axs = plt.subplots(1, weights.shape[0], figsize=(12, 2))
for ax, filt in zip(axs, weights):
    ax.imshow(filt[0], cmap='bwr')
    ax.axis('off')
plt.suptitle('First-layer CNN filters')
plt.show()


## Why CNNs Can Be Translation Equivariant or Invariant

A convolutional filter is reused at every image location. If an edge shifts to the right, the corresponding feature map response shifts to the right. This is called **translation equivariance**.

Pooling and later classifier layers can reduce sensitivity to the exact position of a feature. This can make the final prediction more **translation invariant**.


## Exercise: Compare MLP and CNN

1. Which model has fewer parameters?
2. Which model converges faster?
3. Which model generalizes better on the test set?
4. What do the first-layer CNN filters appear to detect?
5. Why is convolution a useful inductive bias for images?
